# Official VDJdb release: matched ESM evaluations
Run the cells in order in Google Colab. In Runtime → Change runtime type, select a GPU. Upload the prepared `official_release_colab_bundle.zip` from your computer when prompted. These are new experiments on the official release, separate from the older chunk-based results.

The default runs ESM-2 on matched strict and uniform splits with three seeds. ESM-1v / TCR-ESM-style is optional and much larger. GPU availability and runtime limits depend on Colab; save results to Drive.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import io, os, zipfile, subprocess, sys, json
drive.mount('/content/drive')
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Please select exactly one project bundle ZIP.')
uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
if not uploaded_name.lower().endswith('.zip'):
    raise ValueError('Please select a ZIP file.')
with zipfile.ZipFile(io.BytesIO(uploaded_bytes)) as z:
    z.extractall('/content/tcr_official')
os.chdir('/content/tcr_official')
print('Uploaded:', uploaded_name)
print('Working folder:', Path.cwd())


In [ ]:
%pip install -q fair-esm pandas numpy scikit-learn
import torch, platform, importlib.metadata
print('Python:', platform.python_version())
print('Torch:', torch.__version__, 'GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None (slow CPU)')
print('fair-esm:', importlib.metadata.version('fair-esm'))
if not torch.cuda.is_available():
    print('Enable a GPU in Runtime → Change runtime type before continuing if available.')


## Generate frozen sequence embeddings
The same embeddings are used for both splits; classifier scaling is fitted within each training fold. Public pretrained model exposure is a separate limitation from supervised peptide-disjoint splitting.

In [ ]:
split_dir = Path('work/official_2026_06_03/splits')
embed = Path('/content/tcr_official/embeddings/esm2.npz')
cmd = [sys.executable, '-m', 'PhongLu.Modeleval.generate_esm2_embeddings', '--splits', str(split_dir/'matched_strict_five_fold.csv'), str(split_dir/'matched_uniform_five_fold.csv'), '--output', str(embed), '--model', 'esm2_t6_8M_UR50D', '--batch-size', '64']
subprocess.run(cmd, check=True)


## Evaluate matched strict and uniform
The three seeds repeat training on the same fixed splits; they do not measure uncertainty from generating different splits. Training can take time. Results are written to your Drive, so you can recover them if Colab disconnects.

In [ ]:
from datetime import datetime, timezone
run_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out_root = Path('/content/drive/MyDrive/tcr_official_release_esm_runs') / run_stamp
out_root.mkdir(parents=True, exist_ok=False)
environment = {'python': platform.python_version(), 'packages': {name: importlib.metadata.version(name) for name in ('torch', 'fair-esm', 'numpy', 'pandas', 'scikit-learn')}, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, 'esm2_checkpoint_requested': 'esm2_t6_8M_UR50D', 'esm1v_checkpoint_if_enabled': 'esm1v_t33_650M_UR90S_1', 'note': 'Records this new execution, not the historical runs.'}
(out_root / 'environment.json').write_text(json.dumps(environment, indent=2))
for split_name in ('matched_strict', 'matched_uniform'):
    for seed in (13, 42, 123):
        outdir = out_root / f'esm2_{split_name}_seed_{seed}'
        cmd = [sys.executable, '-m', 'PhongLu.Modeleval.train_esm2_classifier', '--splits', str(split_dir/f'{split_name}_five_fold.csv'), '--embeddings', str(embed), '--outdir', str(outdir), '--seed', str(seed), '--architecture', 'simple']
        print('Running:', split_name, 'seed', seed, flush=True)
        subprocess.run(cmd, check=True)
print('Saved to:', out_root)


## Optional: ESM-1v / TCR-ESM-style
This downloads a large model and may take much longer or exceed Colab limits. Set `run_tcr_esm = True` only if sufficient resources are available. This project uses a TCR-ESM-style classifier and does not claim an exact reproduction of the original paper.

In [ ]:
run_tcr_esm = False
if run_tcr_esm:
    embed_tcr = Path('/content/tcr_official/embeddings/esm1v.npz')
    subprocess.run([sys.executable, '-m', 'PhongLu.Modeleval.generate_esm2_embeddings', '--splits', str(split_dir/'matched_strict_five_fold.csv'), str(split_dir/'matched_uniform_five_fold.csv'), '--output', str(embed_tcr), '--model', 'esm1v_t33_650M_UR90S_1', '--batch-size', '8'], check=True)
    for split_name in ('matched_strict', 'matched_uniform'):
        for seed in (13, 42, 123):
            subprocess.run([sys.executable, '-m', 'PhongLu.Modeleval.train_esm2_classifier', '--splits', str(split_dir/f'{split_name}_five_fold.csv'), '--embeddings', str(embed_tcr), '--outdir', str(out_root/f'tcr_esm_{split_name}_seed_{seed}'), '--seed', str(seed), '--architecture', 'tcr-esm'], check=True)


In [ ]:
for path in sorted(out_root.glob('*/summary.json')):
    summary = json.loads(path.read_text())
    print(path.parent.name, 'AUROC:', summary['auroc'], 'AUPRC:', summary['auprc'])
